In [ ]:
# =============================================
# CELL 1: Setup (run once per session)
# =============================================
import sys, subprocess, os

# Kaggle/NVIDIA API keys (use Kaggle Secrets in production)
os.environ["KAGGLE_USERNAME"] = gorrupotunihitha
os.environ["KAGGLE_KEY"] = KGAT_539e26cc5040664e4bfe5dc38eed046a
os.environ["NVIDIA_API_KEY"] = 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

# Install dependencies
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "qiskit>=1.0.0", "qiskit-aer>=0.14.0",
    "pynvml", "pyyaml", "pandas", "kaggle", "requests",
    "--quiet"
])

# Clone repo
if not os.path.exists("openqsim-ai"):
    subprocess.check_call(["git", "clone", "https://github.com/lekkalaharsha/opensim-ai"])
else:
    os.chdir("openqsim-ai")
    subprocess.check_call(["git", "pull", "origin", "main"])
    os.chdir("..")

sys.path.append("openqsim-ai")
print("✅ Setup complete")

In [ ]:
# =============================================
# CELL 2: Run Benchmark Sweep (2-6 hours)
# =============================================
from kaggle import KaggleRunner

runner = KaggleRunner(
    sweep_config_path="openqsim-ai/benchmark/sweep_config_0a.yaml",
    output_dir="/kaggle/working/benchmark_outputs",
    checkpoint_interval=10,
    artifact_interval=50,
    kaggle_dataset="your-username/openqsim-benchmarks",
)

result = runner.run()
print(f"✅ {result.completed_count} records completed")
print(f"   OOM: {result.oom_count} | Errors: {result.error_count}")

In [ ]:
# =============================================
# CELL 3: Assemble Dataset (5 minutes)
# =============================================
from kaggle.dataset_assembler import DatasetAssembler

assembler = DatasetAssembler(
    raw_dir="/kaggle/working/benchmark_outputs/raw",
    dataset_dir="/kaggle/working/benchmark_outputs/datasets/openqsim_v0.1-small",
)
manifest = assembler.assemble()
print(f"✅ Dataset: {manifest.records} records, {manifest.unique_circuits} circuits")

In [ ]:
# =============================================
# CELL 4: Final Push to Kaggle Dataset
# =============================================
from kaggle.api_client import KaggleAPIClient

client = KaggleAPIClient(dataset="your-username/openqsim-benchmarks")
client.push_benchmark_outputs(
    "/kaggle/working/benchmark_outputs",
    message=f"Phase 0A complete: {manifest.records} records"
)
print("✅ Data persisted to Kaggle Dataset")

In [ ]:
# Test NVIDIA NIM advisor on Kaggle
import os, sys
sys.path.append("openqsim-ai")

from backend.llm_advisor import NIMBackendAdvisor

advisor = NIMBackendAdvisor(
    api_key=os.environ["NVIDIA_API_KEY"],
    model="meta/llama-3.1-8b-instruct"
)

fingerprint = {
    "qubits": 20,
    "depth": 40,
    "gate_counts": {"h": 20, "cx": 120},
    "interaction_graph": {"diameter": 5}
}

rec = advisor.recommend_backend(
    qasm="OPENQASM 2.0;\ninclude \"qelib1.inc\";\nqreg q[20];\n...",
    fingerprint=fingerprint
)

print(f"Backend: {rec.recommended_backend}")
print(f"Confidence: {rec.confidence:.2f}")
print(f"Reasoning: {rec.reasoning}")